# StereoCrafter Simple Test Notebook
Edit paths below and run all cells to process 3D video

In [ ]:
import os
import sys

# === EDIT THESE PATHS ===
SOURCE_PATH = "Z:/ScreamTut/Clips"
DEPTH_PATH = "Z:/ScreamTut/Depth"
OUTPUT_PATH = "Z:/ScreamTut/Splat_test"

# Basic settings
MAX_DISP = 40.0
CONVERGENCE = 0.5
PROCESS_LENGTH = -1
FULL_RES = True
LOW_RES = False

# Create directories
os.makedirs(SOURCE_PATH, exist_ok=True)
os.makedirs(DEPTH_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"Source: {SOURCE_PATH}")
print(f"Depth:  {DEPTH_PATH}")
print(f"Output: {OUTPUT_PATH}")

In [ ]:
# Find matching pairs (dry run)
from core.splatting import SplattingController
from core.common.sidecar_manager import SidecarConfigManager

sidecar_manager = SidecarConfigManager()
controller = SplattingController(sidecar_manager=sidecar_manager)

video_list = controller.find_matching_pairs(SOURCE_PATH, DEPTH_PATH)
print(f"Found {len(video_list)} matching video/depth pairs")
for v in video_list:
    print(f"  Source: {v.get('source_video')}")
    print(f"  Depth:  {v.get('depth_map')}")
    print()

In [ ]:
# Run processing
import time
from core.splatting import ProcessingSettings

settings = ProcessingSettings(
    input_source_clips=SOURCE_PATH,
    input_depth_maps=DEPTH_PATH,
    output_splatted=OUTPUT_PATH,
    enable_full_resolution=FULL_RES,
    full_res_batch_size=7,
    enable_low_resolution=LOW_RES,
    low_res_batch_size=13,
    low_res_width=640,
    low_res_height=360,
    process_length=PROCESS_LENGTH,
    zero_disparity_anchor=CONVERGENCE,
    move_to_finished=False,
    output_crf=18,
    output_crf_full=18,
    depth_dilate_size_x=6,
    depth_dilate_size_y=3,
    depth_blur_size_x=1,
    depth_blur_size_y=1,
    depth_gamma=1.0,
    max_disp=MAX_DISP,
    dual_output=True,
)

print("Starting batch processing...")
controller.start_batch(settings)

# Monitor progress - press Stop to interrupt
try:
    while True:
        msg = controller.get_progress()
        if msg:
            print(msg)
            if msg == "finished":
                break
        time.sleep(0.5)
        if not controller.processing_thread or not controller.processing_thread.is_alive():
            break
except KeyboardInterrupt:
    print("\nStopping...")
    controller.stop()
    if controller.processing_thread and controller.processing_thread.is_alive():
        controller.processing_thread.join(timeout=10)
    print("Stopped.")

print("Done!")